# **PINN** Hirota and NLS Equations
## Training Notebook

---

This notebook trains PINNs to solve nonlinear dispersive equations with different penalty configurations on the boundary conditions (BC), as well as different non-adaptive sampling strategies, for different network architectures, training data settings and solution types:

* **Hirota**:
  $$i\,\psi_t
+ 6iR_3 |\psi|^2\psi_x
+ \tfrac{1}{2}D_2\psi_{xx}
+ iD_3\psi_{xxx}
+ R_2|\psi|^2\psi = 0.$$

* **NLS**:
  $$   i\psi_t + \tfrac{1}{2}\psi_{xx} + |\psi|^2\psi = 0 $$


HOW TO RUN:
  1. Run as is: the default case (Hirota N=1 fundamental soliton, Zhou & Yan) uses `data/Bright_soliton.mat`, which ships with the repository.
  2. To train another case, change `equation` and `case` in the configuration cell. `NLS_N1.mat` and `NLS_N2.mat` are also included; the two-soliton files (NLS collision, Hirota breather and Hirota collision) are available on request — see `data/README.md`.
  3. On Colab, mount Google Drive and set `REPO_PATH` in Cell 1 to the folder where the repository was copied.

OUTPUT FILES (per experiment):
  - PINN_results_{exp_id}.npz    -> numerical results and plotting data
  - PINN_metadata_{exp_id}.json  -> experiment configuration and metrics
  - Model_{exp_id}.pt            -> final network weights (for retraining)

THE CODE IS DIRECTLY COMPATIBLE WITH THE POSTPROCESSING CODES

NOTE ON RESIDUALS (f_u, f_v):

  Three residual definitions (real and imaginary components) were used in this project. Cell 3 selects the one that matches `equation` (`'yan'`, `'hirota'` or `'nls'`).

**NOTE ON COLLOCATION POINT GENERATION**:
  scipy.stats.qmc.LatinHypercube is used.
  Compatible with Colab without additional installations.
  
**Implemented strategies:**
- `LHS`    — Latin Hypercube Sampling (baseline)
- `Grid`   — Coherent uniform grid (finite-difference style)
- `Random` — Uniform random sampling (pure Monte Carlo)
- `Sobol`  — Base-2 low-discrepancy quasi-random sequence

**Reference:** Wu et al. (2023). *A comprehensive study of non-adaptive and
residual-based adaptive sampling for physics-informed neural networks.*
Computer Methods in Applied Mechanics and Engineering, 403, 115671.


In [ ]:
# ==============================================================================
# CONFIGURATION — edit the parameters of the experiment here
# ==============================================================================
# DEFAULT: Hirota N=1 fundamental soliton (Zhou & Yan, 2021), which runs with
# the reference data shipped in data/. The other cases need .mat files that
# are available on request (see data/README.md); the training notebook stops
# with a clear message if the selected file is missing.
#
#   equation         case            .mat file                      x-domain   t-domain  in repo
#   'nls'            'fundamental'   NLS_N1.mat                     [-5, 5]    [0, pi/2]   yes
#   'nls'            'bound_state'   NLS_N2.mat                     [-5, 5]    [0, pi/2]   yes
#   'nls'            'collision'     NLS_2soliton_interaction.mat   [0, 20]    [0, 5]      request
#   'yan'            'fundamental'   Bright_soliton.mat             [-10, 10]  [0, 5]      yes
#   'hirota'         'bound_state'   hirota_N_2.mat                 [-5, 5]    [0, 2pi]    request
#   'hirota'         'collision'     N_2_hirota_inter.mat           [-10, 10]  [0, 4]      request
#
# equation = 'yan'    -> Hirota equation, Zhou & Yan (2021) parametrization (alpha=1, beta=0.01)
# equation = 'hirota' -> Hirota equation, Dr. L. Serkin's parametrization (D2, D3, R2, R3) = (1, 0.01, 1, 0.01)
# equation = 'nls'    -> standard NLS equation
# Domains follow Table 3.2 of the thesis.

equation     = 'yan'                     # 'yan' | 'hirota' | 'nls'
case         = 'fundamental'             # 'fundamental' | 'bound_state' (breather) | 'collision'
architecture = [2, 80, 80, 80, 80, 2]    # [input, hidden..., output]  (4 hidden layers x 80 neurons)
sampling_strategies = ['LHS', 'Sobol', 'Grid', 'Random']   # one experiment per strategy
bc_mode      = 1                         # boundary-condition mode:
                                         #   0, 1 : No BC
                                         #      2 : Val Only (periodic in psi)
                                         #      3 : Der1 Only (periodic in d_x psi)
                                         #      4 : Val + Der1
                                         #      5 : Val + Der1 + Der2
seed         = 1234                      # random seed (does not apply to Grid)
n_adam       = 5000                      # Adam epochs
n_lbfgs      = 10000                     # maximum L-BFGS iterations
lr           = 0.001                     # Adam learning rate

import numpy as np

CASES = {
    # (equation, case): (filename, lb=[x_min, t_min], ub=[x_max, t_max], short tag)
    ('nls',    'fundamental'): ('NLS_N1.mat',                   [-5.0, 0.0],  [5.0, np.pi/2], 'NLS_N1'),
    ('nls',    'bound_state'): ('NLS_N2.mat',                   [-5.0, 0.0],  [5.0, np.pi/2], 'NLS_N2Bound'),
    ('nls',    'collision'):   ('NLS_2soliton_interaction.mat', [0.0, 0.0],   [20.0, 5.0],    'NLS_N2Col'),
    ('yan',    'fundamental'): ('Bright_soliton.mat',           [-10.0, 0.0], [10.0, 5.0],    'Yan_N1'),
    ('hirota', 'bound_state'): ('hirota_N_2.mat',               [-5.0, 0.0],  [5.0, 2*np.pi], 'Hir_N2Bound'),
    ('hirota', 'collision'):   ('N_2_hirota_inter.mat',         [-10.0, 0.0], [10.0, 4.0],    'Hir_N2Col'),
}
if (equation, case) not in CASES:
    raise ValueError(f'No reference data for equation={equation!r}, case={case!r}. '
                     f'Valid combinations: {sorted(CASES)}')

data_file, lb_case, ub_case, case_tag = CASES[(equation, case)]
lb_case = np.array(lb_case)
ub_case = np.array(ub_case)

arch_tag = f'{len(architecture) - 2}x{architecture[1]}'
bc_tag   = {0: 'NoBC', 1: 'NoBC', 2: 'ValOnly', 3: 'Der1Only',
            4: 'ValDer1', 5: 'ValDer1Der2'}[bc_mode]
BATCH_ID = f'{case_tag}_{arch_tag}_{bc_tag}_Sampling'   # outputs/<BATCH_ID>/ and results/<BATCH_ID>/


def make_exp_id(strategy):
    """Experiment identifier; it is also the stem of the output file names."""
    return f'{case_tag}_{arch_tag}_{strategy}_{n_adam / 1000:g}kAdam_{n_lbfgs / 1000:g}kFGS_{bc_tag}'


print(f'Case     : equation={equation!r}, case={case!r} -> {data_file}')
print(f'Domain   : x in [{lb_case[0]:g}, {ub_case[0]:g}], t in [{lb_case[1]:g}, {ub_case[1]:.4g}]')
print(f'Batch ID : {BATCH_ID}')


## CELL 1 — SETUP AND LIBRARIES

In [ ]:
# ==============================================================================
# CELL 1 — SETUP AND LIBRARIES
# ==============================================================================

import os
import sys
import time
import json
import numpy as np
import scipy.io
from scipy.interpolate import griddata
from scipy.stats import qmc
import torch
import torch.nn as nn
import torch.optim as optim

# ------------------------------------------------------------------------
# Requirement: this notebook is designed to run on Google Colab
# with a GPU (T4 or other). See README.md, section "Installation".
# ------------------------------------------------------------------------
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Adjust to the path where the repo was cloned/copied inside Drive:
    REPO_PATH = '/content/drive/MyDrive/PINN_NLS_Hirota'
else:
    # Local execution (VS Code, Terminal, Jupyter, etc.). Assumes the notebook is run from notebooks/
    REPO_PATH = os.path.abspath(os.path.join(os.getcwd(), '..'))

sys.path.append(os.path.join(REPO_PATH, 'src'))
DATA_PATH = os.path.join(REPO_PATH, 'data')

from sampling_strategies import get_sampling

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA version: {torch.version.cuda}')
    print(f'CUDNN version: {torch.backends.cudnn.version()}')
    print(f'Number of CUDA devices: {torch.cuda.device_count()}')
else:
    print('WARNING: running on CPU. Training will be slow.')
print(f'Current Path: {REPO_PATH}')

## CELL 2 — NETWORK ARCHITECTURE

In [ ]:
# ==============================================================================
# CELL 2 — NETWORK ARCHITECTURE
# ==============================================================================

class NeuralNetwork(nn.Module):
    """
    Feed-forward neural network with Tanh activation and Xavier initialization.

    Parameters
    ----------
    layers : list of int
        Number of neurons per layer, including input and output.
        Example: [2, 40, 40, 40, 2] -> input (x,t), 3 hidden layers, output (u,v).
    """
    def __init__(self, layers):
        super(NeuralNetwork, self).__init__()
        self.depth      = len(layers) - 1
        self.activation = nn.Tanh()
        layer_list = []
        for i in range(self.depth):
            layer_list.append(nn.Linear(layers[i], layers[i + 1]))
            nn.init.xavier_normal_(layer_list[i].weight)
            nn.init.zeros_(layer_list[i].bias)
        self.layers = nn.ModuleList(layer_list)

    def forward(self, x):
        for i in range(self.depth - 1):
            x = self.activation(self.layers[i](x))
        x = self.layers[-1](x)
        return x


## CELL 3 — PHYSICS-INFORMED NEURAL NETWORK (PINN)

In [ ]:
# ==============================================================================
# CELL 3 — PHYSICS-INFORMED NEURAL NETWORK (PINN)
# ==============================================================================
#
#   NOTE ON THE LOSS HISTORY:
#   self.history records EVERY call to loss_fn() (Adam + L-BFGS).
#   Keys:
#     'total' → total loss (IC + BC + Res)
#     'ic'    → initial condition loss
#     'bc'    → boundary condition loss (fields + derivatives)
#     'res'   → PDE residual loss
#     self.adam_iters marks the exact index where Adam ends.
#
#   NOTE ON BOUNDARY CONDITIONS:
#   Periodic conditions are imposed on the fields u, v and their spatial
#   derivatives du/dx, dv/dx at x = lb[0] and x = ub[0].

class PhysicsInformedNN:
    """
    PINN for the NLS and Hirota equations with optional periodic conditions.

    Params
    ----------
    x0 : np.ndarray (N0_eff, 1)
        Spatial coordinates of the initial condition.
        N0_eff may differ from N0 in the Grid strategy.
    u0, v0 : np.ndarray (N0_eff, 1)
        Real and imaginary parts of q at t=0.
    tb : np.ndarray (Nb_eff, 1)
        Time instants for the boundary condition.
        Nb_eff may differ from Nb in the Grid strategy.
    X_f : np.ndarray (Nf_eff, 2)
        Interior collocation points [x, t].
    layers : list of int
        Network architecture.
    lb, ub : np.ndarray (2,)
        Domain bounds [x_min, t_min] and [x_max, t_max].
    X_u_train : np.ndarray (N0_eff+2*Nb_eff, 2)
        Training points (IC + boundaries), for reference.
    lbfgs_iter : int
        Maximum number of L-BFGS iterations.
    equation : str
        Residual definition: 'yan' (Hirota, Zhou & Yan), 'hirota'
        (Hirota, Dr. Serkin's parametrization) or 'nls'.
    """

    def __init__(self, x0, u0, v0, tb, X_f, layers, lb, ub, X_u_train, lbfgs_iter, lr,
                 bc_mode, equation):

        if equation not in ('yan', 'hirota', 'nls'):
            raise ValueError(f"equation must be 'yan', 'hirota' or 'nls', got {equation!r}")
        self.equation = equation

        # --- Initial condition (t = 0) ---
        self.x0 = torch.tensor(x0,                    dtype=torch.float32).to(device)
        self.t0 = torch.tensor(np.zeros_like(x0),     dtype=torch.float32).to(device)
        self.u0 = torch.tensor(u0,                    dtype=torch.float32).to(device)
        self.v0 = torch.tensor(v0,                    dtype=torch.float32).to(device)

        # --- Boundary conditions (x = lb[0] and x = ub[0]) ---
        self.x_lb = torch.tensor(lb[0] * np.ones_like(tb), dtype=torch.float32).to(device)
        self.t_lb = torch.tensor(tb,                        dtype=torch.float32).to(device)
        self.x_ub = torch.tensor(ub[0] * np.ones_like(tb), dtype=torch.float32).to(device)
        self.t_ub = torch.tensor(tb,                        dtype=torch.float32).to(device)

        # --- Interior collocation points ---
        self.x_f = torch.tensor(X_f[:, 0:1], dtype=torch.float32).to(device)
        self.t_f = torch.tensor(X_f[:, 1:2], dtype=torch.float32).to(device)

        # --- Domain bounds (for internal normalization to [-1,1]) ---
        self.lb = torch.tensor(lb, dtype=torch.float32).to(device)
        self.ub = torch.tensor(ub, dtype=torch.float32).to(device)

        # --- Reference to the training points ---
        self.X_u_train = X_u_train

        # --- BC mode ---
        self.bc_mode = bc_mode

        # --- Model ---
        self.layers = layers
        self.model  = NeuralNetwork(layers).to(device)

        # --- Optimizers ---
        self.lr = lr
        self.optimizer_adam = optim.Adam(self.model.parameters(), lr=lr)

        self.lbfgs_iter = lbfgs_iter
        self.optimizer_lbfgs = optim.LBFGS(
            self.model.parameters(),
            max_iter        = lbfgs_iter,
            max_eval        = lbfgs_iter,
            history_size    = 50,
            tolerance_grad  = 1e-9,
            tolerance_change= 1e-12,
            line_search_fn  = 'strong_wolfe'
        )

        # --- Unified history (Adam + L-BFGS) ---
        self.history = {'total': [], 'ic': [], 'bc': [], 'res': []}
        self.adam_iters  = 0
        self._lbfgs_iter = 0

    # --------------------------------------------------------------------------
    def net_uv(self, x, t):
        """
        Network evaluation: (x, t) -> (u, v).
        The inputs are normalized to [-1, 1] before going through the network.
        """
        X      = torch.cat([x, t], dim=1)
        X_norm = 2.0 * (X - self.lb) / (self.ub - self.lb) - 1.0
        uv     = self.model(X_norm)
        return uv[:, 0:1], uv[:, 1:2]

    # --------------------------------------------------------------------------
    def net_f_uv(self, x, t):
        """
        PDE residuals via automatic differentiation.

        The residual follows self.equation; third derivatives are
        only computed for the Hirota equation.

        """
        x = x.clone().detach().requires_grad_(True)
        t = t.clone().detach().requires_grad_(True)
        u, v = self.net_uv(x, t)

        u_t  = torch.autograd.grad(u,   t, grad_outputs=torch.ones_like(u),
                                   retain_graph=True, create_graph=True)[0]
        u_x  = torch.autograd.grad(u,   x, grad_outputs=torch.ones_like(u),
                                   retain_graph=True, create_graph=True)[0]
        u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x),
                                   retain_graph=True, create_graph=True)[0]
        v_t  = torch.autograd.grad(v,   t, grad_outputs=torch.ones_like(v),
                                   retain_graph=True, create_graph=True)[0]
        v_x  = torch.autograd.grad(v,   x, grad_outputs=torch.ones_like(v),
                                   retain_graph=True, create_graph=True)[0]
        v_xx = torch.autograd.grad(v_x, x, grad_outputs=torch.ones_like(v_x),
                                   retain_graph=True, create_graph=True)[0]
        if self.equation != 'nls':
            u_xxx = torch.autograd.grad(u_xx, x, grad_outputs=torch.ones_like(u_xx),
                                        retain_graph=True, create_graph=True)[0]
            v_xxx = torch.autograd.grad(v_xx, x, grad_outputs=torch.ones_like(v_xx),
                                        retain_graph=True, create_graph=True)[0]

        # ------------------------------------------------------------------
        # HIROTA EQUATION, ZHOU & YAN (2021) PAPER  (equation = 'yan')
        # ------------------------------------------------------------------
        if self.equation == 'yan':
            alpha = 1.0
            beta  = 0.01
            f_u = -v_t + alpha*(u_xx + 2*(u**2 + v**2)*u) - beta*(v_xxx + 6*(u**2 + v**2)*v_x)
            f_v =  u_t + alpha*(v_xx + 2*(u**2 + v**2)*v) + beta*(u_xxx + 6*(u**2 + v**2)*u_x)
            return f_u, f_v

        # ------------------------------------------------------------------
        # NLS EQUATION, SATSUMA & YAJIMA BOUND STATE  (equation = 'nls')
        # ------------------------------------------------------------------
        if self.equation == 'nls':
            f_u = u_t + 0.5 * v_xx + (u**2 + v**2) * v
            f_v = v_t - 0.5 * u_xx - (u**2 + v**2) * u
            return f_u, f_v

        # ------------------------------------------------------------------
        # HIROTA EQUATION, DR. L. SERKIN'S CODE  (equation = 'hirota')
        # NOTE: the sign convention (f_u = fake_v, f_v = -fake_u) comes
        # directly from the source code provided by Dr. Serkin; it is kept
        # as is for the breather and soliton collision cases.
        # ------------------------------------------------------------------
        D2 = 1.0
        D3 = 0.01
        R2 = 1.0
        R3 = 0.01

        fake_u = -v_t + 0.5*D2*u_xx + R2*(u**2 + v**2)*u - D3*v_xxx - 6*R3*(u**2 + v**2)*v_x
        fake_v =  u_t + 0.5*D2*v_xx + R2*(u**2 + v**2)*v + D3*u_xxx + 6*R3*(u**2 + v**2)*u_x

        f_u = fake_v
        f_v = -(fake_u)

        return f_u, f_v

    def loss_fn(self):
        """
        Total loss = IC + BC (according to bc_mode) + PDE residuals.
        Records each component in self.history on every call.
        """
        # Initial condition
        u_pred0, v_pred0 = self.net_uv(self.x0, self.t0)
        loss_ic = (torch.mean((self.u0 - u_pred0)**2) +
                   torch.mean((self.v0 - v_pred0)**2))

        # Boundary condition
        loss_bc = torch.tensor(0.0, device=device)

        if self.bc_mode > 1:
            u_lb, v_lb = self.net_uv(self.x_lb, self.t_lb)
            u_ub, v_ub = self.net_uv(self.x_ub, self.t_ub)

        # Modes 2, 4, 5: periodicity in value
        if self.bc_mode in [2, 4, 5]:
            loss_bc = loss_bc + (torch.mean((u_lb - u_ub)**2) +
                                 torch.mean((v_lb - v_ub)**2))

        # Modes 3, 4, 5: periodicity in the 1st derivative
        if self.bc_mode in [3, 4, 5]:
            x_lb_g = self.x_lb.clone().detach().requires_grad_(True)
            u_lb_g, v_lb_g = self.net_uv(x_lb_g, self.t_lb)
            u_x_lb = torch.autograd.grad(u_lb_g, x_lb_g,
                                         grad_outputs=torch.ones_like(u_lb_g),
                                         create_graph=True)[0]
            v_x_lb = torch.autograd.grad(v_lb_g, x_lb_g,
                                         grad_outputs=torch.ones_like(v_lb_g),
                                         create_graph=True)[0]

            x_ub_g = self.x_ub.clone().detach().requires_grad_(True)
            u_ub_g, v_ub_g = self.net_uv(x_ub_g, self.t_ub)
            u_x_ub = torch.autograd.grad(u_ub_g, x_ub_g,
                                         grad_outputs=torch.ones_like(u_ub_g),
                                         create_graph=True)[0]
            v_x_ub = torch.autograd.grad(v_ub_g, x_ub_g,
                                         grad_outputs=torch.ones_like(v_ub_g),
                                         create_graph=True)[0]

            loss_bc = loss_bc + (torch.mean((u_x_lb - u_x_ub)**2) +
                                 torch.mean((v_x_lb - v_x_ub)**2))

        # Mode 5:  periodicity in the 2nd derivative
        if self.bc_mode == 5:
            u_xx_lb = torch.autograd.grad(u_x_lb, x_lb_g,
                                          grad_outputs=torch.ones_like(u_x_lb),
                                          create_graph=True)[0]
            v_xx_lb = torch.autograd.grad(v_x_lb, x_lb_g,
                                          grad_outputs=torch.ones_like(v_x_lb),
                                          create_graph=True)[0]
            u_xx_ub = torch.autograd.grad(u_x_ub, x_ub_g,
                                          grad_outputs=torch.ones_like(u_x_ub),
                                          create_graph=True)[0]
            v_xx_ub = torch.autograd.grad(v_x_ub, x_ub_g,
                                          grad_outputs=torch.ones_like(v_x_ub),
                                          create_graph=True)[0]
            loss_bc = loss_bc + (torch.mean((u_xx_lb - u_xx_ub)**2) +
                                 torch.mean((v_xx_lb - v_xx_ub)**2))

        # PDE residuals
        f_u, f_v = self.net_f_uv(self.x_f, self.t_f)
        loss_res = torch.mean(f_u**2) + torch.mean(f_v**2)

        # Total
        loss = loss_ic + loss_bc + loss_res

        # Logging
        self.history['total'].append(loss.item())
        self.history['ic'].append(loss_ic.item())
        self.history['bc'].append(loss_bc.item())
        self.history['res'].append(loss_res.item())

        return loss

    def closure(self):
        """Closure required by L-BFGS."""
        self.optimizer_lbfgs.zero_grad()
        loss = self.loss_fn()
        loss.backward()
        self._lbfgs_iter += 1
        if self._lbfgs_iter % 500 == 0:
            print(f'   L-BFGS Iter {self._lbfgs_iter}: Loss = {loss.item():.5e}')
        return loss

    def train(self, epochs):
        """Two-phase training: Adam -> L-BFGS."""
        self.model.train()
        print('   Starting Adam...')
        for epoch in range(epochs):
            self.optimizer_adam.zero_grad()
            loss = self.loss_fn()
            loss.backward()
            self.optimizer_adam.step()
            if epoch % 1000 == 0:
                print(f'   Adam Epoch {epoch}: Loss = {loss.item():.5e}')
        self.adam_iters  = len(self.history['total'])
        print('   Starting L-BFGS...')
        self._lbfgs_iter = 0
        self.optimizer_lbfgs.step(self.closure)
        print('   Training finished.')

    def predict(self, X_star):
        """Evaluates the network at X_star = [x, t]. Returns u, v, f_u, f_v."""
        self.model.eval()
        x_eval = torch.tensor(X_star[:, 0:1], dtype=torch.float32).to(device)
        t_eval = torch.tensor(X_star[:, 1:2], dtype=torch.float32).to(device)
        with torch.no_grad():
            u, v = self.net_uv(x_eval, t_eval)
        with torch.enable_grad():
            x_f = torch.tensor(X_star[:, 0:1], dtype=torch.float32).to(device).requires_grad_(True)
            t_f = torch.tensor(X_star[:, 1:2], dtype=torch.float32).to(device).requires_grad_(True)
            f_u, f_v = self.net_f_uv(x_f, t_f)
        return (u.cpu().numpy(), v.cpu().numpy(),
                f_u.detach().cpu().numpy(), f_v.detach().cpu().numpy())


## CELL 4 — EXPERIMENT CONFIGURATION

In [ ]:
# ==============================================================================
# CELL 4 — SAMPLING EXPERIMENT CONFIGURATION
# ==============================================================================
#
# Each entry defines a complete experiment.
# Required fields:
#   exp_id      : str   — unique identifier (output file name)
#   sampling    : str   — strategy: 'LHS' | 'Grid' | 'Random' | 'Sobol'
#   filename    : str   — .mat file under REPO_PATH/data
#   batch_id    : str   — batch name (common subfolder in outputs/)
#   lb, ub      : array — domain bounds [x_min, t_min] / [x_max, t_max] (see the thesis for the domains per solution)
#   layers      : list  — architecture [2, n1, n2, ..., 2]
#   N0          : int   — requested IC points (in Grid it is determined by the grid)
#   Nb          : int   — requested BC points (in Grid it is determined by the grid)
#   Nf          : int   — interior collocation points
#   Adam_epochs : int   — Adam epochs
#   lbfgs_iter  : int   — maximum L-BFGS iterations
#   seed        : int   — random seed (does not apply to Grid)
#
# BOUNDARY MODES:
#   0, 1: No BC
#      2: Val Only
#      3: Der1 Only
#      4: Val + Der1
#      5: Val + Der1 + Der2
#
# NOTE ON Grid:
#   N0 and Nb are ignored — they are determined by Nx_g and Nt_g of the grid.
#   The effective number is reported to the console and saved in the JSON.
#
# FILE NAMES:
#   exp_id is built by make_exp_id() in the configuration cell and includes
#   the case, architecture, sampling strategy, budget and BC mode.
#   Example: 'Yan_N1_4x80_LHS_5kAdam_10kFGS_NoBC'
#
# SOLUTION TYPES (selected through `equation` and `case` in the configuration cell):
#   NLS:
#     Fundamental: 'NLS_N1.mat'
#     Breather:    'NLS_N2.mat'
#     Collision:   'NLS_2soliton_interaction.mat'
#   Hirota:
#     Fundamental (Zhou):   'Bright_soliton.mat'
#     Breather (L. Serkin): 'hirota_N_2.mat'
#     Collision (L. Serkin): 'N_2_hirota_inter.mat'

outputs_root = os.path.join(REPO_PATH, 'outputs')  # results root

# --- Check that the reference data for the selected case is available -------
data_path = os.path.join(DATA_PATH, data_file)
if not os.path.exists(data_path):
    raise FileNotFoundError(
        f'{data_path} not found.\n'
        f'The case equation={equation!r}, case={case!r} needs {data_file}, which is not '
        f'redistributed in this repository. It is available on request: see data/README.md. '
        f"The cases that run out of the box are ('yan', 'fundamental'), ('nls', 'fundamental') "
        f"and ('nls', 'bound_state')."
    )

experiments = [
    {
        'exp_id':      make_exp_id(strategy),
        'sampling':    strategy,
        'bc_mode':     bc_mode,
        'equation':    equation,
        'filename':    data_file,
        'batch_id':    BATCH_ID,
        'lb':          lb_case,
        'ub':          ub_case,
        'layers':      architecture,
        'N0':          50,      # ignored in Grid; determined by the grid
        'Nb':          50,      # ignored in Grid; determined by the grid
        'Nf':          10000,
        'Adam_epochs': n_adam,
        'lbfgs_iter':  n_lbfgs,
        'seed':        seed,    # ignored in Grid (deterministic)
        'lr':          lr,
    }
    for strategy in sampling_strategies
]

for exp in experiments:
    print(exp['exp_id'])


## CELL 5 — MAIN EXPERIMENT LOOP

In [ ]:
# ==============================================================================
# CELL 5 — MAIN EXPERIMENT LOOP
# ==============================================================================
# For each experiment: load the reference data, generate the training points
# with get_sampling(), train (Adam + L-BFGS), evaluate and save
# .npz + .json + .pt. The JSON stores the strategy and the effective N
# (relevant for Grid).
# ==============================================================================

for exp in experiments:

    # --- Experiment parameters ---
    exp_id      = exp['exp_id']
    sampling    = exp['sampling']
    filename    = exp['filename']
    batch_id    = exp['batch_id']
    outputs_dir = os.path.join(outputs_root, batch_id) + '/'
    lb          = exp['lb']
    ub          = exp['ub']
    layers      = exp['layers']
    N0          = exp['N0']
    Nb          = exp['Nb']
    Nf          = exp['Nf']
    Adam_epochs = exp['Adam_epochs']
    lbfgs_iter  = exp['lbfgs_iter']
    seed        = exp.get('seed', 1234)
    lr          = exp['lr']
    bc_mode     = exp['bc_mode']
    eq          = exp['equation']

    # --- Output paths ---
    if not os.path.exists(outputs_dir):
        os.makedirs(outputs_dir)

    npz_file  = f'{outputs_dir}PINN_results_{exp_id}.npz'
    json_file = f'{outputs_dir}PINN_metadata_{exp_id}.json'
    pt_file   = f'{outputs_dir}Model_{exp_id}.pt'

    # --- Checkpoint: skip if it already exists ---
    if os.path.exists(npz_file) and os.path.exists(json_file):
        print(f'--> Skipping {exp_id} (results already exist in {outputs_dir}).')
        continue

    print(f'\n{"="*60}')
    print(f'Experiment  : {exp_id}')
    print(f'Sampling    : {sampling}')
    print(f'Equation    : {eq}')
    print(f'File        : {filename}')
    print(f'Domain      : lb={lb}, ub={ub}')
    print(f'Network     : {layers}')
    print(f'Optimizer   : Adam={Adam_epochs} epochs, L-BFGS={lbfgs_iter} iter.')
    print(f'{"="*60}')

    # ==========================================================================
    # 1. DATA LOADING
    # ==========================================================================
    file_path = os.path.join(REPO_PATH, 'data', filename)
    if not os.path.exists(file_path):
        print(f'ERROR: {file_path} not found. Skipping.')
        continue

    data    = scipy.io.loadmat(file_path)
    t_all   = data['t_values'].flatten()[:, None]   # (Nt, 1)
    x_all   = data['x_values'].flatten()[:, None]   # (Nx, 1)
    Exact   = data['q_values']                       # (Nx, Nt) complex
    Exact_u = np.real(Exact)                         # (Nx, Nt)
    Exact_v = np.imag(Exact)                         # (Nx, Nt)
    Exact_h = np.sqrt(Exact_u**2 + Exact_v**2)      # (Nx, Nt)

    X, T   = np.meshgrid(x_all, t_all)              # (Nt, Nx)
    X_star = np.hstack((X.flatten()[:, None],
                        T.flatten()[:, None]))        # (Nt*Nx, 2)

    U_star = Exact_u.T   # (Nt, Nx)
    V_star = Exact_v.T   # (Nt, Nx)
    H_star = Exact_h.T   # (Nt, Nx)

    u_star = Exact_u.T.flatten()[:, None]   # (Nt*Nx, 1)
    v_star = Exact_v.T.flatten()[:, None]
    h_star = Exact_h.T.flatten()[:, None]

    # ==========================================================================
    # 2. TRAINING POINT GENERATION
    # ==========================================================================
    # get_sampling() always returns the same dictionary structure,
    # regardless of the chosen strategy.

    # --- Per-experiment reproducibility (for sampling) ---
    torch.manual_seed(seed)
    np.random.seed(seed)

    points    = get_sampling(
        strategy = sampling,
        lb       = lb,
        ub       = ub,
        Nf       = Nf,
        N0       = N0,
        Nb       = Nb,
        Exact_u  = Exact_u,
        Exact_v  = Exact_v,
        x_all    = x_all,
        t_all    = t_all,
        seed     = seed,
    )
    x0        = points['x0']
    u0_train  = points['u0_train']
    v0_train  = points['v0_train']
    tb_train  = points['tb_train']
    X_f_train = points['X_f_train']
    N0_eff    = points['N0_eff']    # effective (may differ in Grid)
    Nb_eff    = points['Nb_eff']
    Nf_eff    = points['Nf_eff']

    # Consolidated training points (IC + boundaries) for reference
    X0_meta   = np.concatenate((x0,                   0 * x0),         axis=1)
    X_lb_meta = np.concatenate((0*tb_train + lb[0],   tb_train),       axis=1)
    X_ub_meta = np.concatenate((0*tb_train + ub[0],   tb_train),       axis=1)
    X_u_train_meta = np.vstack([X0_meta, X_lb_meta, X_ub_meta])

    # ==========================================================================
    # 3. TRAINING
    # ==========================================================================
    torch.cuda.empty_cache()

    # --- Per-experiment reproducibility (weight initialization) ---
    # We fix the seed again so that the network restarts from the same generator state
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = PhysicsInformedNN(
        x0, u0_train, v0_train, tb_train, X_f_train,
        layers, lb, ub, X_u_train_meta, lbfgs_iter, lr, bc_mode, eq
    )

    start_time = time.time()
    model.train(epochs=Adam_epochs)
    elapsed = time.time() - start_time
    print(f'\n   Training time: {elapsed:.2f} s')

    # ==========================================================================
    # 4. PREDICTION AND METRICS
    # ==========================================================================
    u_pred, v_pred, f_u_pred, f_v_pred = model.predict(X_star)
    h_pred = np.sqrt(u_pred**2 + v_pred**2)

    error_u = np.linalg.norm(u_star - u_pred, 2) / np.linalg.norm(u_star, 2)
    error_v = np.linalg.norm(v_star - v_pred, 2) / np.linalg.norm(v_star, 2)
    error_h = np.linalg.norm(h_star - h_pred, 2) / np.linalg.norm(h_star, 2)
    print(f'   Relative L2 error u: {error_u:.3e} | v: {error_v:.3e} | |q|: {error_h:.3e}')

    U_pred_grid = griddata(X_star, u_pred.flatten(),   (X, T), method='cubic')
    V_pred_grid = griddata(X_star, v_pred.flatten(),   (X, T), method='cubic')
    H_pred_grid = griddata(X_star, h_pred.flatten(),   (X, T), method='cubic')
    FU_pred     = griddata(X_star, f_u_pred.flatten(), (X, T), method='cubic')
    FV_pred     = griddata(X_star, f_v_pred.flatten(), (X, T), method='cubic')

    # ==========================================================================
    # 5. SAVING RESULTS
    # ==========================================================================
    np.savez_compressed(npz_file,

        # Axes and grid
        x            = x_all,
        t            = t_all,
        X            = X,
        T            = T,

        # Exact solution
        Exact_u      = Exact_u,
        Exact_v      = Exact_v,
        Exact_h      = Exact_h,
        U_star       = U_star,
        V_star       = V_star,
        H_star       = H_star,

        # Prediction on the grid
        U_pred       = U_pred_grid,
        V_pred       = V_pred_grid,
        H_pred       = H_pred_grid,

        # Raw prediction
        u_pred_raw   = u_pred,
        v_pred_raw   = v_pred,

        # Residuals
        f_u_pred_raw = f_u_pred,
        f_v_pred_raw = f_v_pred,
        FU_pred      = FU_pred,
        FV_pred      = FV_pred,

        # Training points
        X_u_train    = X_u_train_meta,
        X_f_train    = X_f_train,

        # Metrics
        error_u      = error_u,
        error_v      = error_v,
        error_h      = error_h,

        # History
        loss_total   = np.array(model.history['total']),
        loss_ic      = np.array(model.history['ic']),
        loss_bc      = np.array(model.history['bc']),
        loss_res     = np.array(model.history['res']),
        adam_iters   = np.array([model.adam_iters]),

        # Domain
        lb           = lb,
        ub           = ub,

        # Time
        elapsed_time = elapsed,
    )

    # JSON metadata
    metadata = {
        # Identification
        'id':           exp_id,
        'filename':     filename,
        'equation':     eq,
        'sampling':     sampling,           # sampling strategy

        # Architecture
        'layers':       layers,

        # Training points
        # N0, Nb, Nf: REQUESTED values in the experiments dictionary.
        # N0_eff, Nb_eff, Nf_eff: EFFECTIVE values generated by get_sampling().
        # In Grid they may differ; in the rest they are equal.
        'N0':           int(N0),
        'Nb':           int(Nb),
        'Nf':           int(Nf),
        'N0_eff':       int(N0_eff),
        'Nb_eff':       int(Nb_eff),
        'Nf_eff':       int(Nf_eff),

        # Domain
        'lb':           lb.tolist(),
        'ub':           ub.tolist(),

        # Boundaries:
        'bc_mode':      bc_mode,

        # Optimization
        'adam_epochs':           Adam_epochs,
        'lbfgs_iter':            lbfgs_iter,
        'lr':                    lr,
        'adam_iters_recorded':   model.adam_iters,
        'lbfgs_iters_recorded':  len(model.history['total']) - model.adam_iters,
        'total_iters_recorded':  len(model.history['total']),

        # L2 metrics
        'error_u':      float(error_u),
        'error_v':      float(error_v),
        'error_h':      float(error_h),

        # Environment
        'training_time_sec': float(elapsed),
        'framework':         'PyTorch',
        'device':            str(device),
        'seed':              int(seed),
    }

    with open(json_file, 'w') as f:
        json.dump(metadata, f, indent=4)

    torch.save(model.model.state_dict(), pt_file)

    print(f'\n   Saved: {npz_file}')
    print(f'   Saved: {json_file}')
    print(f'   Saved: {pt_file}')

    del model, u_pred, v_pred, h_pred, f_u_pred, f_v_pred
    del U_pred_grid, V_pred_grid, H_pred_grid, FU_pred, FV_pred
    torch.cuda.empty_cache()

print('\n--- All sampling experiments completed ---')
